# SCI1070 — Semaine 12 : un seul objet, Puissance 4, tenu de bout en bout

| Partie | Ce que vous y faites |
| --- | --- |
| Échauffement | le modèle de jeu en cinq fonctions, et trois vérifications |
| A | la recherche, **testée sans réseau entraîné** |
| B | l'auto-jeu complet, et le tournoi des points de contrôle |
| C | trois constantes, et la mesure qui commande chacune |
| D | lecture et analyse multi-agent, **aucun entraînement** |

> **Votre critère de progrès cette semaine n'est pas une perte, c'est un tournoi.**
> Si vous déclarez « j'arrête quand la perte descend sous telle valeur », **vous
> n'arrêterez jamais**. **C'est la première fois que ce cours vous demande d'ignorer
> une courbe de perte, et c'est volontaire.**

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "torch"], check=True)

import collections
import copy
import math

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Les quatre constantes de la section 12.4.
N_LIGNES, N_COLONNES = 6, 7
N_ALIGNES = 4
BITS_HAUTEUR = 3          # de quoi coder une hauteur de colonne de 0 à 6

print(f"{N_LIGNES} x {N_COLONNES}, {N_ALIGNES} alignés, {BITS_HAUTEUR} bits de hauteur")

---
# Échauffement : le modèle du jeu

**Tout le reste repose dessus.** Cinq fonctions : encodage, décodage, coups
possibles, jouer un coup, afficher.

In [ ]:
...        # À COMPLÉTER : les cinq fonctions de la section 12.4

afficher(plateau_vide)
print("coups possibles :", coups_possibles(plateau_vide))

### Les trois vérifications, dans cet ordre

In [ ]:
...        # À COMPLÉTER : 1. ALLER-RETOUR sur mille positions tirées au hasard

In [ ]:
...        # À COMPLÉTER : 2. LE MATCH NUL : une grille pleine sans victoire

In [ ]:
...        # À COMPLÉTER : 3. L'HORIZON, dérivé et non recopié

---
# Volet A : la recherche, testée sans réseau entraîné

In [ ]:
class ReseauJeu(nn.Module):
    """Deux plans de 6 x 7, un tronc résiduel, deux têtes."""

    def __init__(self, n_filtres=48):
        super().__init__()
        ...        # À COMPLÉTER : deux plans en entrée, deux têtes en sortie


def encoder_tenseur(plateau, joueur):
    """Deux plans : mes jetons, ses jetons. L'ÉCHANGE des plans encode le joueur."""
    t = torch.zeros(2, N_LIGNES, N_COLONNES)
    for c in range(N_COLONNES):
        for l, jeton in enumerate(plateau[c]):
            t[0 if jeton == joueur else 1, l, c] = 1.0
    return t

In [ ]:
class Recherche:
    def __init__(self, c_puct=1.0, alpha_dirichlet=0.3, utilite_amputee=False, graine=0):
        self.c_puct = c_puct
        self.alpha_dirichlet = alpha_dirichlet
        self.utilite_amputee = utilite_amputee      # pour l'étape 3
        self.rng = np.random.default_rng(graine)
        self.visites = {}
        self.valeurs = {}
        self.priors = {}

    def _utilite(self, code):
        """La bonne utilité, et la version amputée sous le drapeau."""
        n = self.visites[code]
        q = np.divide(self.valeurs[code], n, out=np.zeros_like(n), where=n > 0)
        ...        # À COMPLÉTER : sans racine carrée et sans constante pour la version amputée

    def _expanser(self, plateau, tour, reseau, bruiter=False):
        """Évalue une feuille, enregistre ses priors, et rend sa valeur."""
        code = encoder(plateau)
        legaux = coups_possibles(plateau)
        with torch.no_grad():
            logits, v = reseau(encoder_tenseur(plateau, tour).unsqueeze(0))
        priors = F.softmax(logits[0], dim=0).numpy()
        masque = np.zeros(N_COLONNES)
        masque[legaux] = 1.0
        priors = priors * masque                   # masquer AVANT l'argmax
        priors = priors / (priors.sum() + 1e-8)
        if bruiter:
            bruit = self.rng.dirichlet([self.alpha_dirichlet] * N_COLONNES)
            priors = 0.75 * priors + 0.25 * bruit * masque
        self.visites[code] = np.zeros(N_COLONNES)
        self.valeurs[code] = np.zeros(N_COLONNES)
        self.priors[code] = priors
        return float(v)

    def rechercher(self, plateau, joueur, reseau, n_recherches, racine_bruitee=False):
        ...        # À COMPLÉTER : sélection, expansion, évaluation, remontée AVEC INVERSION DE SIGNE
        return self.visites[encoder(plateau)]

### Étape 2 — trois tests qui valent mieux qu'une heure d'entraînement

In [ ]:
torch.manual_seed(0)
reseau_neuf = ...        # À COMPLÉTER : le réseau du volet B, NON entraîné

...        # À COMPLÉTER : LE TEST DU TOTAL : après n recherches, la somme des comptages vaut n

In [ ]:
...        # À COMPLÉTER : LE TEST DE LA VICTOIRE IMMÉDIATE

In [ ]:
...        # À COMPLÉTER : LE TEST DU SIGNE, celui qui attrape l'erreur de la section 12.3

### Étape 3, l'expérience la plus instructive du laboratoire

In [ ]:
for amputee in [False, True]:
    r = Recherche(utilite_amputee=amputee, graine=0)
    v = r.rechercher(plateau_vide, 0, reseau_neuf, 200)
    distinctes = int((v >= 10).sum())
    etiquette = "utilité amputée" if amputee else "utilité correcte"
    print(f"{etiquette:18s} : {distinctes} colonne(s) visitée(s) au moins dix fois")

Sans la racine carrée ni la constante, le terme d'exploration ne décroît plus quand
une colonne est visitée : la recherche **s'enferme** sur son premier choix. C'est
l'avertissement de la section 12.3, reproduit dans votre code.

---
# Volet B : l'auto-jeu, et une perte qui ne descendra pas

In [ ]:
...        # À COMPLÉTER : la vérification d'invariance : la même position, vue des deux camps

In [ ]:
def jouer_une_partie(reseau, c_puct=1.0, tau_coups=10, n_recherches=10, graine=0):
    """Rend la liste des triplets (etat, politique_cible, resultat)."""
    ...        # À COMPLÉTER : PIÈGE Nº1 : si vous ne retournez pas le résultat en remontant l'historique, la moitié de vos cibles de valeur ont le signe inverse.

In [ ]:
def porte(apprenti, meilleur, n_parties=40, seuil=0.60, n_recherches=10, graine=0):
    """Rend (taux_de_victoire, promu). La COPIE des poids est le piège nº3."""
    ...        # À COMPLÉTER : PIÈGE Nº3 : promouvoir en gardant une RÉFÉRENCE au même objet rend les deux réseaux identiques, et la porte s'ouvre désormais au hasard.

In [ ]:
...        # À COMPLÉTER : la boucle d'auto-jeu, et les trois courbes à prédire AVANT

In [ ]:
...        # À COMPLÉTER : étape 5 : le tournoi des points de contrôle

---
# Volet C : trois constantes, et ce que chacune commande

In [ ]:
...        # À COMPLÉTER : étape 1 : c_puct sur trois valeurs séparées par un facteur dix

In [ ]:
rng = np.random.default_rng(0)
for alpha in [0.03, 0.3]:
    tirages = rng.dirichlet(np.full(N_COLONNES, alpha), size=1000)
    print(f"alpha = {alpha:5.2f} : poids maximal moyen {tirages.max(axis=1).mean():.3f}")
print("\n-> à 0,03 le vecteur est presque unitaire : il DÉSIGNE une colonne")
print("   au lieu de perturber doucement les sept, ce qui n'est pas le rôle")
print("   du bruit de racine.")

In [ ]:
...        # À COMPLÉTER : étape 3 : le compteur de parties tronquées par l'horizon, qui doit valoir zéro

**Étape 4 — une phrase par constante.**

* \(c_{\text{puct}}\) commande la **largeur** de la recherche, et se règle en
  regardant le nombre de colonnes réellement explorées à la racine.
* \(\alpha_{\mathrm{Dir}}\) commande la **forme du bruit de racine**, et se règle en
  regardant le poids maximal moyen d'un tirage, **sans entraîner quoi que ce soit**.
* L'horizon commande la **proportion de parties dont la cible de valeur correspond à
  un résultat réel**, et se règle en comptant les parties tronquées.

---
# Volet D : lecture et analyse d'un monde multi-agent

**Aucun calcul lourd** : de la lecture, deux divisions et de l'écriture.

**Question 1, l'interface.** Une boucle multi-agent itère sur un **dictionnaire
d'agents** et rend cinq dictionnaires au lieu de cinq valeurs. Une enveloppe
ordinaire suppose un `step()` qui rend un seul quintuplet : elle n'a pas où brancher
la clé d'agent, elle ne sait pas quoi faire d'un agent qui disparaît, et elle ne peut
pas fournir un espace d'observations unique.

**Question 2, la population volatile.** Au pas 37, l'agent mort possède encore une
entrée dans les cinq dictionnaires, avec son drapeau de terminaison à vrai : c'est sa
**dernière** transition, et elle doit entrer dans le tampon avec `termine = True`. Au
pas 38, sa clé a **disparu** des cinq dictionnaires : il n'y a rien à ranger, et
chercher sa clé lève. Itérer sur la liste des agents sans la figer modifie la
collection pendant l'itération, donc lève ou saute des agents.

**Question 3, l'observation.** Encoder l'observation **relativement au groupe de
l'agent** — mes alliés dans un plan, mes adversaires dans l'autre — rend une politique
partagée possible. **C'est exactement le procédé de l'encodage à deux plans de la
première moitié de cette semaine.** Ce que cette politique partagée **interdit** :
deux agents d'un même groupe dans la même situation locale ne peuvent pas se
spécialiser, donc aucune division des rôles.

**Question 4, l'arithmétique.** Avec une santé initiale \(H\), un coût par pas \(c\),
une santé rendue par proie \(g\) et une récompense par pas \(r\), un score de test
\(S\) sur \(T\) pas donne un nombre de proies \(\approx (cT - H)/g\). Le calcul **ne
tient que si la récompense par pas est la seule source de score**, c'est-à-dire si
chasser ne rapporte pas de récompense directe. Avec un score de 60 au lieu de 82, le
nombre baisse proportionnellement mais la conclusion **ne change pas de nature** :
c'est toujours une inférence indirecte, pas une mesure.

**Question 5, et c'est la question qui compte.** Ce qui est **certain** : les agents
ont été entraînés sur une récompense individuelle. Ce qui est **observé** : ils
chassent souvent par deux. L'affirmation « ils ont appris à collaborer » ajoute une
**intention** que ni l'entraînement ni l'observation ne contiennent : chasser par
deux peut être le simple résultat de deux agents attirés indépendamment par la même
proie. **La question unique qui tranche** : le comportement survit-il si l'on retire
à chaque agent toute perception de ses congénères ? S'il survit, il n'y avait pas de
coordination.

**Question 6.** Deux courbes plates n'ont pas la même cause. En auto-jeu, l'adversaire
progresse avec vous : une courbe plate est **attendue** et ne dit rien, et la réponse
utile est un tournoi contre des points de contrôle anciens. Avec deux groupes
entraînés simultanément, une courbe plate peut signaler un équilibre, mais aussi
qu'aucun des deux n'apprend ; la réponse utile est de **geler** un groupe et de
mesurer l'autre contre lui.

---
# Il n'y a pas de solution courte cette semaine

Les bibliothèques de haut niveau sont construites autour d'une interface **à un seul
agent**. La première moitié de la semaine a besoin d'un modèle de jeu **interrogeable
depuis un état arbitraire**, ce que cette interface interdit.

In [ ]:
# Deux adversaires de reference, a battre avant de se rejouir de quoi que ce soit.
#
# 1. Le joueur aleatoire.
# 2. La recherche seule, avec votre reseau NON entraine.

...        # À COMPLÉTER : les deux matchs

> **Le second adversaire est le plus intéressant des deux.** Une recherche
> arborescente, même avec un réseau complètement aléatoire, **joue déjà quelque
> chose** : elle détecte les victoires et les défaites immédiates, parce que
> celles-ci sont des états **terminaux**, évalués par le résultat réel de la partie
> et non par le réseau. **Si vous ne mesurez pas cela, vous attribuerez à votre
> réseau un mérite qui revient à votre arbre.**